In [3]:
import os
if not os.path.exists("generate_data.py"):
    from google.colab import files
    print("Select generate_data.py from your computer...")
    files.upload()
!python generate_data.py


generated: customers=1200 products=40 orders_raw=13084 tickets=164


In [4]:
import sqlite3
import numpy as np
import pandas as pd

os.makedirs("data/clean", exist_ok=True); os.makedirs("sql", exist_ok=True)
cust = pd.read_csv("data/raw/customers.csv")
prod = pd.read_csv("data/raw/products.csv")
orders = pd.read_csv("data/raw/orders.csv")
print(cust.shape, prod.shape, orders.shape)

(1200, 5) (40, 4) (13084, 9)


In [6]:
for name, df in [("customers", cust), ("products", prod), ("orders", orders)]:
    print(f"\n{name}: shape={df.shape}")
    print(pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum(),
                        "missing_pct": (df.isna().mean() * 100).round(2)}))


customers: shape=(1200, 5)
               dtype  missing  missing_pct
customer_id   object        0         0.00
city          object        0         0.00
signup_date   object        0         0.00
age          float64       61         5.08
gender        object       30         2.50

products: shape=(40, 4)
               dtype  missing  missing_pct
product_id    object        0          0.0
product_name  object        0          0.0
category      object        0          0.0
price_inr      int64        0          0.0

orders: shape=(13084, 9)
                    dtype  missing  missing_pct
order_id           object        0         0.00
customer_id        object        0         0.00
product_id         object        0         0.00
quantity            int64        0         0.00
order_ts           object        0         0.00
status             object        0         0.00
discount_pct        int64        0         0.00
order_amount       object       53         0.41
delivery_minutes

In [10]:
amt_num = pd.to_numeric(orders["order_amount"], errors="coerce")
audit = {
    "raw orders rows": len(orders),                                   # ref 13084
    "exact duplicate rows": int(orders.duplicated().sum()),           # ref 119
    "pending rows": int((orders["status"] == "pending").sum()),       # ref 656
    "distinct order_id": int(orders["order_id"].nunique()),           # ref 12309
    "non-numeric order_amount": int(amt_num.isna().sum()),            # ref 120
    "negative delivery_minutes": int((orders["delivery_minutes"] < 0).sum()),   # ref 106
    "blank delivery_minutes": int(orders["delivery_minutes"].isna().sum()),     # ref 2541
    "blank age": int(cust["age"].isna().sum()),                       # ref 61
    "age outside 10-100": int(((cust["age"] < 10) | (cust["age"] > 100)).sum()),# ref 6
    "distinct raw city spellings": int(cust["city"].nunique()),       # ref 24
    "distinct non-blank gender spellings": int(cust["gender"].dropna().nunique()), # ref 10
    "blank gender": int(cust["gender"].isna().sum()),                 # ref 30
}
pd.Series(audit, name="value").to_frame()

,value
raw orders rows,13084
exact duplicate rows,119
pending rows,656
distinct order_id,12309
non-numeric order_amount,120
negative delivery_minutes,106
blank delivery_minutes,2541
blank age,61
age outside 10-100,6
distinct raw city spellings,24


In [11]:
c = cust.copy()
c["city"] = c["city"].str.strip().str.title()
gmap = {"m": "male", "male": "male", "f": "female", "female": "female"}
c["gender"] = c["gender"].fillna("").str.strip().str.lower().map(gmap).fillna("unknown")

bad = (c["age"] < 10) | (c["age"] > 100)
print("impossible ages set to missing:", int(bad.sum()))
c.loc[bad, "age"] = np.nan
print("ages to fill with city median:", int(c["age"].isna().sum()))
c["age"] = c["age"].fillna(c.groupby("city")["age"].transform("median")).astype(int)
c["signup_date"] = pd.to_datetime(c["signup_date"])

print(c["city"].value_counts().to_dict())      # Pune 227, Delhi 210, Bengaluru 197, Chennai 191, Hyderabad 191, Mumbai 184
print(c["gender"].value_counts().to_dict())    # female 610, male 560, unknown 30
print("missing ages:", c["age"].isna().sum(), "| age range:", c["age"].min(), "-", c["age"].max())

impossible ages set to missing: 6
ages to fill with city median: 67
{'Pune': 227, 'Delhi': 210, 'Bengaluru': 197, 'Chennai': 191, 'Hyderabad': 191, 'Mumbai': 184}
{'female': 610, 'male': 560, 'unknown': 30}
missing ages: 0 | age range: 18 - 60


In [12]:
o = orders.copy()
log = {}

# Rule 1: datetime
o["order_ts"] = pd.to_datetime(o["order_ts"])

# Rule 2: exact duplicates
n = len(o); o = o.drop_duplicates(); log["exact duplicates removed"] = n - len(o)

# Rule 3: per order_id keep the LATEST order_ts (drops the earlier 'pending' record)
n = len(o)
o = o.sort_values(["order_id", "order_ts"]).drop_duplicates("order_id", keep="last")
log["older records removed (keep latest per order_id)"] = n - len(o)

# Rule 4: amount -> numeric, fill from price * qty * (1 - discount/100)
o["order_amount"] = pd.to_numeric(o["order_amount"], errors="coerce")
o = o.merge(prod[["product_id", "category", "price_inr"]], on="product_id", how="left")
miss = o["order_amount"].isna()
log["order_amount filled from product price"] = int(miss.sum())
calc = (o["price_inr"] * o["quantity"] * (1 - o["discount_pct"] / 100)).round(2)
o.loc[miss, "order_amount"] = calc[miss]

# Rule 5: negative delivery -> missing; completed -> category median (transform); cancelled stay missing
neg = o["delivery_minutes"] < 0
log["negative delivery_minutes set to missing"] = int(neg.sum())
o.loc[neg, "delivery_minutes"] = np.nan
done = o["status"] == "completed"
cat_median = o[done].groupby("category")["delivery_minutes"].transform("median")
need = done & o["delivery_minutes"].isna()
log["completed delivery_minutes imputed (category median)"] = int(need.sum())
o.loc[need, "delivery_minutes"] = cat_median[need]

o = o.drop(columns=["category", "price_inr"]).sort_values("order_id").reset_index(drop=True)
print(pd.Series(log, name="rows changed").to_frame())

                                                    rows changed
exact duplicates removed                                     119
older records removed (keep latest per order_id)             656
order_amount filled from product price                       112
negative delivery_minutes set to missing                     105
completed delivery_minutes imputed (category me...           774


In [13]:
comp = o[o.status == "completed"]
print("rows:", len(o))                                              # 12309
print(o.status.value_counts().to_dict())                            # completed 11110, cancelled 1199
print("missing order_amount:", o.order_amount.isna().sum())         # 0
print("missing delivery (completed):", comp.delivery_minutes.isna().sum(),
      "| (cancelled):", o[o.status == "cancelled"].delivery_minutes.isna().sum())   # 0 | 1199
print("completed revenue INR:", round(comp.order_amount.sum(), 2))  # 2998668.8

rows: 12309
{'completed': 11110, 'cancelled': 1199}
missing order_amount: 0
missing delivery (completed): 0 | (cancelled): 1199
completed revenue INR: 2998668.8


In [15]:
c_out = c.copy(); c_out["signup_date"] = c_out["signup_date"].dt.strftime("%Y-%m-%d")
o_out = o.copy(); o_out["order_ts"] = o_out["order_ts"].dt.strftime("%Y-%m-%d %H:%M:%S")
c_out.to_csv("data/clean/customers.csv", index=False)
prod.to_csv("data/clean/products.csv", index=False)
o_out.to_csv("data/clean/orders.csv", index=False)

if os.path.exists("data/instamart.db"): os.remove("data/instamart.db")
with sqlite3.connect("data/instamart.db") as con:
    c_out.to_sql("customers", con, index=False)
    prod.to_sql("products", con, index=False)
    o_out.to_sql("orders", con, index=False)
print("saved CSVs and data/instamart.db")

saved CSVs and data/instamart.db


In [16]:
cc = pd.read_csv("data/clean/customers.csv"); oo = pd.read_csv("data/clean/orders.csv")
dd = oo[oo.status == "completed"]
checks = {
 "city counts": cc.city.value_counts().to_dict() == {"Pune":227,"Delhi":210,"Bengaluru":197,"Chennai":191,"Hyderabad":191,"Mumbai":184},
 "six cities": cc.city.nunique() == 6,
 "genders": cc.gender.value_counts().to_dict() == {"female":610,"male":560,"unknown":30},
 "no missing ages": cc.age.isna().sum() == 0,
 "ages 18-60": cc.age.between(18, 60).all(),
 "12309 unique orders": len(oo) == 12309 and oo.order_id.is_unique,
 "11110 completed / 1199 cancelled": (oo.status=="completed").sum()==11110 and (oo.status=="cancelled").sum()==1199,
 "no pending": (oo.status == "pending").sum() == 0,
 "no missing amount": oo.order_amount.isna().sum() == 0,
 "no missing delivery (completed)": dd.delivery_minutes.isna().sum() == 0,
 "1199 missing delivery (cancelled)": oo[oo.status=="cancelled"].delivery_minutes.isna().sum() == 1199,
 "revenue 2,998,668.80": round(dd.order_amount.sum(), 2) == 2998668.80,
}
for k, v in checks.items(): print("PASS" if v else "FAIL", k)
assert all(checks.values())
print("ALL CHECKPOINTS PASSED")

PASS city counts
PASS six cities
PASS genders
PASS no missing ages
PASS ages 18-60
PASS 12309 unique orders
PASS 11110 completed / 1199 cancelled
PASS no pending
PASS no missing amount
PASS no missing delivery (completed)
PASS 1199 missing delivery (cancelled)
PASS revenue 2,998,668.80
ALL CHECKPOINTS PASSED


In [18]:
Q = {}
Q["Q1"] = """SELECT customer_id, SUM(order_amount) AS total_big_orders
FROM orders WHERE status = 'completed' AND order_amount > 500
GROUP BY customer_id HAVING SUM(order_amount) > 8000"""
Q["Q2"] = """SELECT c.city, p.category, ROUND(SUM(o.order_amount), 2) AS revenue
FROM orders o JOIN customers c ON c.customer_id = o.customer_id
JOIN products p ON p.product_id = o.product_id
WHERE o.status = 'completed' GROUP BY c.city, p.category ORDER BY revenue DESC"""
Q["Q3"] = """SELECT product_id, product_name, category, price_inr FROM products
WHERE price_inr > (SELECT AVG(price_inr) FROM products) ORDER BY price_inr DESC"""
Q["Q4"] = """SELECT c.customer_id, c.city FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id AND o.status = 'completed'
WHERE o.order_id IS NULL ORDER BY c.customer_id"""
Q["Q5"] = """WITH spend AS (SELECT customer_id, SUM(order_amount) AS total_spend
    FROM orders WHERE status = 'completed' GROUP BY customer_id),
avg_spend AS (SELECT AVG(total_spend) AS avg_total FROM spend)
SELECT s.customer_id, ROUND(s.total_spend, 2) AS total_spend, ROUND(a.avg_total, 2) AS avg_total
FROM spend s CROSS JOIN avg_spend a WHERE s.total_spend > a.avg_total ORDER BY s.total_spend DESC"""
Q["Q6"] = """WITH spend AS (SELECT c.customer_id, c.city, ROUND(SUM(o.order_amount), 2) AS total_spend
    FROM orders o JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.status = 'completed' GROUP BY c.customer_id, c.city)
SELECT customer_id, city, total_spend,
  RANK()       OVER (PARTITION BY city ORDER BY total_spend DESC) AS rnk,
  DENSE_RANK() OVER (PARTITION BY city ORDER BY total_spend DESC) AS dense_rnk,
  ROW_NUMBER() OVER (PARTITION BY city ORDER BY total_spend DESC) AS row_num
FROM spend ORDER BY city, row_num"""
Q["Q7"] = """WITH daily AS (SELECT SUBSTR(order_ts, 1, 10) AS day, ROUND(SUM(order_amount), 2) AS revenue
    FROM orders WHERE status = 'completed' GROUP BY SUBSTR(order_ts, 1, 10))
SELECT day, revenue, ROUND(SUM(revenue) OVER (ORDER BY day), 2) AS cumulative_revenue
FROM daily ORDER BY day"""

# save to sql/queries.sql
with open("sql/queries.sql", "w") as f:
    for k, q in Q.items():
        f.write(f"-- {k}\n{q};\n\n")

con = sqlite3.connect("data/instamart.db")
R = {k: pd.read_sql_query(q, con) for k, q in Q.items()}

In [19]:
print("Q1 customers:", len(R["Q1"]))                                       # 30
print("Q2 rows:", len(R["Q2"]), "| total:", round(R["Q2"].revenue.sum(), 2))   # 30 | 2998668.8
print("Q2 largest cell:", R["Q2"].iloc[0].to_dict())                       # Bengaluru / Electronics 233909.45
print("Q3 products:", len(R["Q3"]))                                        # 11
print("Q4 customers:", len(R["Q4"]))                                       # 25
print("Q5 customers:", len(R["Q5"]), "| average:", R["Q5"].avg_total.iloc[0])   # 550 | 2552.06
q6 = R["Q6"]
print("Q6 top per city:"); display(q6[q6.row_num == 1][["city", "customer_id", "total_spend"]])
print("Q6 rows with RANK != ROW_NUMBER:", int((q6.rnk != q6.row_num).sum()))   # 8
print("Q7 rows:", len(R["Q7"]), "| last cumulative:", R["Q7"].cumulative_revenue.iloc[-1])   # 183 | 2998668.8

Q1 customers: 30
Q2 rows: 30 | total: 2998668.8
Q2 largest cell: {'city': 'Bengaluru', 'category': 'Electronics', 'revenue': 233909.45}
Q3 products: 11
Q4 customers: 25
Q5 customers: 550 | average: 2552.06
Q6 top per city:


,city,customer_id,total_spend
0,Bengaluru,C0082,12856.20
192,Chennai,C0930,10151.40
379,Delhi,C0291,13742.60
584,Hyderabad,C0743,12730.05
771,Mumbai,C0693,14766.55
954,Pune,C0426,10377.10


Q6 rows with RANK != ROW_NUMBER: 8
Q7 rows: 183 | last cumulative: 2998668.8


In [20]:
oc = pd.read_sql_query("SELECT * FROM orders WHERE status='completed'", con)
cu = pd.read_sql_query("SELECT * FROM customers", con)
pr = pd.read_sql_query("SELECT * FROM products", con)
m = oc.merge(cu, on="customer_id").merge(pr, on="product_id")

pv = pd.pivot_table(m, values="order_amount", index="city", columns="category", aggfunc="sum", margins=True)
display(pv.round(2))
print("pivot grand total:", round(pv.loc["All", "All"], 2), "| SQL Q2 total:", round(R["Q2"].revenue.sum(), 2),
      "| equal:", abs(pv.loc["All", "All"] - R["Q2"].revenue.sum()) < 0.01)

agg = m.groupby("city").agg(total=("order_amount", "sum"), mean=("order_amount", "mean"),
                            count=("order_amount", "count")).round(2)
agg

category,Dairy,Electronics,Grocery,Personal Care,Snacks,All
city,,,,,,
Bengaluru,87075.05,233909.45,110915.65,20193.85,71549.40,523643.40
Chennai,78301.80,195508.80,93817.45,27675.40,68325.70,463629.15
Delhi,117271.70,158549.10,112658.05,23471.50,90879.45,502829.80
Hyderabad,83731.00,228913.10,109475.40,24800.05,81670.35,528589.90
Mumbai,75209.85,214738.40,78970.15,38018.00,72122.10,479058.50
Pune,90349.55,187629.25,108456.60,34275.15,80207.50,500918.05
All,531938.95,1219248.10,614293.30,168433.95,464754.50,2998668.80


pivot grand total: 2998668.8 | SQL Q2 total: 2998668.8 | equal: True


,total,mean,count
city,,,
Bengaluru,523643.40,276.47,1894
Chennai,463629.15,272.72,1700
Delhi,502829.80,243.62,2064
Hyderabad,528589.90,284.95,1855
Mumbai,479058.50,297.00,1613
Pune,500918.05,252.48,1984


In [21]:
a = oc["order_amount"]
q1, q3 = a.quantile(.25), a.quantile(.75); iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"mean {a.mean():.2f} | median {a.median():.2f} | skewness {a.skew():.2f}")
print(f"Q1 {q1:.2f} | Q3 {q3:.2f} | lower fence {lo:.2f} | upper fence {hi:.2f}")
print(f"outliers below lower fence: {(a < lo).sum()} | above upper fence: {(a > hi).sum()}")

corr = oc[["order_amount", "discount_pct", "quantity", "delivery_minutes"]].corr()
display(corr.round(3))
pairs = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
best = pairs.abs().idxmax()
print("largest |correlation| pair:", best, round(pairs[best], 3))

mean 269.91 | median 135.00 | skewness 4.61
Q1 64.80 | Q3 297.00 | lower fence -283.50 | upper fence 645.30
outliers below lower fence: 0 | above upper fence: 897


,order_amount,discount_pct,quantity,delivery_minutes
order_amount,1.000,-0.095,0.087,-0.003
discount_pct,-0.095,1.000,-0.000,-0.010
quantity,0.087,-0.000,1.000,-0.013
delivery_minutes,-0.003,-0.010,-0.013,1.000


largest |correlation| pair: ('order_amount', 'discount_pct') -0.095
